# [LEGACY PROTOTYPE - UNVERIFIED]
**WARNING**: This notebook is an exploratory prototype created during early manuscript drafting.
- It contains unexecuted cells, mock datasets, or synthetic demonstrative outputs.
- **None** of the results or cells in this notebook should be treated as experimental evidence or verified baseline outputs.
- For active, reproducible pipelines, refer to , , and .
---


# SesoFix: Orthographic Harmonization in Sesotho
## High-Performance Experimental Runbook (Optimized for NVIDIA A100 GPU on Google Colab)

This notebook contains all core experiments for **SesoFix**, reframed for **Orthographic Harmonization** (conversions from South African Sesotho to Lesotho Sesotho orthography).

### Hardware Acceleration (NVIDIA A100)
This notebook is configured to push GPU usage to the limit by leveraging:
1. **Base-sized Models**: Fine-tuning `google/byt5-base` and `google/mt5-base` (582M parameters).
2. **Large Batch Sizes**: `per_device_train_batch_size=64` (fully utilizing A100's HBM2 memory).
3. **High-Performance Precisions**: **BF16 mixed precision** and **TF32 tensor core calculations** dynamically enabled.
4. **Data Loader Optimizations**: Pinned memory and 4 loader workers.

---

### Setup Instructions
1. Select the **A100 GPU** runtime in Google Colab (`Runtime -> Change runtime type -> Hardware accelerator -> A100 GPU`).
2. Run the environment setup cell below to install dependencies.

In [ ]:
# 1. Clone repository (if running in Colab environment)
import os
import sys

if not os.path.exists('SesoFix') and not os.path.exists('scripts'):
    print("Cloning SesoFix repository...")
    !git clone https://github.com/OmondiKevin/SesoFix.git
    %cd SesoFix
else:
    print("Already in SesoFix repository directory.")

# 2. Install dependencies
print("Installing core dependencies...")
!pip install -q -r requirements.txt
!pip install -q evaluate sacrebleu jiwer matplotlib pandas openpyxl

# 3. Verify GPU and print A100 configuration status
import torch
print("\n--- GPU Hardware Verification ---")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"BF16 Supported: {torch.cuda.is_bf16_supported()}")
    # Enable TF32 for PyTorch matrix multiplications (A100 optimization)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    print("TF32 tensor cores enabled.")
else:
    print("WARNING: Running on CPU. Base models training will be extremely slow.")

## Section 1: Curation of Clean, Leakage-Free Dataset Splits

In this section, we prepare the dataset splits. To prevent **train-test contamination (data leakage)**, we ensure that:
1. **Validation and Test splits** contain strictly real-world texts (0% synthetic data).
2. **Synthetic data** (generated by rule-based systems) is isolated entirely to the **Training split**.

We will load a dataset containing the `is_synthetic` flag and split it using our updated splitting logic.

In [ ]:
import pandas as pd
from scripts.data_preprocessing import load_data_from_csv, split_dataset
from example import create_sample_data

# Generate the mock CSV containing the is_synthetic column
csv_path = create_sample_data()

# Load the dataset
dataset = load_data_from_csv(csv_path)

# Split the dataset preventing leakage (synthetic_col='is_synthetic')
dataset_dict = split_dataset(dataset, train_ratio=0.6, val_ratio=0.2, test_ratio=0.2, synthetic_col='is_synthetic')

# Print split statistics to verify leakage isolation
print("\n--- Split Verification ---")
print(f"Total Rows: {len(dataset)}")
for split_name in ['train', 'validation', 'test']:
    split_dataset_part = dataset_dict[split_name]
    num_rows = len(split_dataset_part)
    num_synthetic = sum(split_dataset_part['is_synthetic'])
    num_real = num_rows - num_synthetic
    print(f"Split: {split_name.capitalize()}")
    print(f"  Total Rows: {num_rows}")
    print(f"  Real Rows: {num_real}")
    print(f"  Synthetic Rows: {num_synthetic} (Ratio: {num_synthetic/num_rows*100:.1f}%)")

## Section 2: Core Model Comparison (ByT5-Base vs. mT5-Base)

In this section, we fine-tune and compare:
1. **ByT5-Base**: A token-free, byte-level Seq2Seq model (582M parameters).
2. **mT5-Base**: A subword-tokenized multilingual Seq2Seq model (582M parameters).

Both models are trained with a batch size of **64** and mixed precision (`bf16` or `fp16`) to fully leverage the speed of the A100 GPU. We will evaluate them on the clean test set.

In [ ]:
# Train ByT5-Base model
!python3 scripts/train_model.py \
  --model_name "google/byt5-base" \
  --sa_file "data/processed/sample_data.csv" \
  --data_format csv \
  --sa_col "south_african" \
  --ls_col "lesotho" \
  --synthetic_col "is_synthetic" \
  --num_train_epochs 5 \
  --train_batch_size 64 \
  --eval_batch_size 64 \
  --output_dir "./models/byt5_base"

In [ ]:
# Train mT5-Base model
!python3 scripts/train_model.py \
  --model_name "google/mt5-base" \
  --sa_file "data/processed/sample_data.csv" \
  --data_format csv \
  --sa_col "south_african" \
  --ls_col "lesotho" \
  --synthetic_col "is_synthetic" \
  --num_train_epochs 5 \
  --train_batch_size 64 \
  --eval_batch_size 64 \
  --output_dir "./models/mt5_base"

In [ ]:
# Evaluate ByT5-Base predictions on the clean test set
print("Evaluating ByT5-Base...")
!python3 scripts/evaluate.py \
  --model_dir "./models/byt5_base" \
  --sa_file "data/processed/sample_data.csv" \
  --data_format csv \
  --sa_col "south_african" \
  --ls_col "lesotho" \
  --output_file "data/output/byt5_base_predictions.csv" \
  --batch_size 64

# Evaluate mT5-Base predictions on the clean test set
print("\nEvaluating mT5-Base...")
!python3 scripts/evaluate.py \
  --model_dir "./models/mt5_base" \
  --sa_file "data/processed/sample_data.csv" \
  --data_format csv \
  --sa_col "south_african" \
  --ls_col "lesotho" \
  --output_file "data/output/mt5_base_predictions.csv" \
  --batch_size 64

## Section 3: Low-Data Scaling Study on ByT5-Base

Here we conduct the low-data scaling study. We fine-tune **ByT5-Base** on different percentages of the training split ($5\%$, $10\%$, $25\%$, $50\%$, and $100\%$) and evaluate each run on the clean validation/test splits. Finally, we plot the metrics to demonstrate data scaling dynamics.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
from datasets import Dataset, DatasetDict
from scripts.data_preprocessing import load_data_from_csv, split_dataset
from scripts.train_model import Seq2SeqTrainer, DataCollatorForSeq2Seq
from scripts.model_config import load_byt5_model, get_training_args

# Load data and prepare splits
print("Loading dataset...")
dataset = load_data_from_csv("data/processed/sample_data.csv")

# Inject is_synthetic if not present (sample data has it)
if 'is_synthetic' not in dataset.column_names:
    # Set alternating rows as synthetic for mock run
    is_synth_list = [i % 2 == 0 for i in range(len(dataset))]
    dataset = dataset.add_column("is_synthetic", is_synth_list)

dataset_dict = split_dataset(dataset, train_ratio=0.6, val_ratio=0.2, test_ratio=0.2, synthetic_col='is_synthetic')

fractions = [0.05, 0.10, 0.25, 0.50, 1.00]
losses = []

# We define a function to train a subset
def train_fraction(fraction, train_split, val_split, test_split):
    subset_size = max(int(len(train_split) * fraction), 1)
    print(f"\n==========================================")
    print(f"Training on {fraction*100}% data ({subset_size} rows)...")
    print(f"==========================================")
    
    train_subset = train_split.select(range(subset_size))
    
    # Load base model & tokenizer
    model, tokenizer = load_byt5_model("google/byt5-base")
    
    # Prepare datasets (simplified tokenization helper)
    def preprocess_local(examples):
        inputs = [f"normalise: {text}" for text in examples['source']]
        targets = examples['target']
        model_inputs = tokenizer(inputs, max_length=64, padding="max_length", truncation=True)
        with tokenizer.as_target_tokenizer():
            labels = tokenizer(targets, max_length=64, padding="max_length", truncation=True)
        model_inputs["labels"] = labels["input_ids"]
        return model_inputs
        
    local_dict = DatasetDict({
        'train': train_subset,
        'validation': val_split,
        'test': test_split
    })
    
    tokenized = local_dict.map(preprocess_local, batched=True, remove_columns=local_dict["train"].column_names)
    
    # Training args optimized for speed in this scaling loop
    training_args = get_training_args(
        output_dir=f"./models/scaling_{fraction}",
        num_train_epochs=3,
        per_device_train_batch_size=64,
        per_device_eval_batch_size=64,
        warmup_steps=0,
        weight_decay=0.01,
        logging_dir=f"./logs/scaling_{fraction}",
        logging_steps=1,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
    )
    
    data_collator = DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        model=model,
        padding="max_length",
        max_length=64
    )
    
    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=tokenized["train"],
        eval_dataset=tokenized["validation"],
        tokenizer=tokenizer,
        data_collator=data_collator,
    )
    
    trainer.train()
    eval_res = trainer.evaluate(tokenized["test"])
    print(f"Evaluation Loss for {fraction*100}%: {eval_res['eval_loss']}")
    return eval_res['eval_loss']

# Run the study
for f in fractions:
    loss = train_fraction(f, dataset_dict['train'], dataset_dict['validation'], dataset_dict['test'])
    losses.append(loss)

# Plotting the scaling curve
plt.figure(figsize=(8, 5))
plt.plot([f*100 for f in fractions], losses, marker='o', linewidth=2, color='darkblue')
plt.title("ByT5-Base Loss Scaling Study (Sesotho Harmonization)")
plt.xlabel("Percentage of Training Data (%)")
plt.ylabel("Test Loss ↓")
plt.grid(True)
plt.savefig("scaling_loss_curve.png")
plt.show()

## Section 4: Downstream Translation Impact Experiment

To demonstrate the value of SesoFix in downstream applications, we evaluate how orthographic harmonization improves Sesotho-to-English translation quality. 
We translate South African Sesotho text directly using a downstream translation model (`Helsinki-NLP/opus-mt-nso-en` from Hugging Face), and compare the translation BLEU scores for:
1. **Raw South African Sesotho** (unnormalized).
2. **SesoFix Normalized Sesotho** (harmonized to Lesotho orthography).

In [ ]:
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
import pandas as pd
import evaluate
import os

# 1. Load fine-tuned ByT5-Base model (from our training output)
model_path = "./models/byt5_base"
if os.path.exists(model_path):
    print(f"Loading custom SesoFix model from {model_path}...")
    sesofix_tokenizer = AutoTokenizer.from_pretrained(model_path)
    sesofix_model = AutoModelForSeq2SeqLM.from_pretrained(model_path)
else:
    print("Custom model not found. Using pre-trained google/byt5-base for dry run...")
    sesofix_tokenizer = AutoTokenizer.from_pretrained("google/byt5-base")
    sesofix_model = AutoModelForSeq2SeqLM.from_pretrained("google/byt5-base")

# Load downstream translation model (Sesotho to English)
print("Loading downstream translator model Helsinki-NLP/opus-mt-nso-en...")
translator = pipeline("translation", model="Helsinki-NLP/opus-mt-nso-en")

def run_sesofix(text):
    inputs = sesofix_tokenizer(f"normalise: {text}", return_tensors="pt")
    outputs = sesofix_model.generate(**inputs, max_length=128)
    return sesofix_tokenizer.decode(outputs[0], skip_special_tokens=True)

# Sample sentences for evaluation
sa_sentences = [
    "Ke rata ho bala dibuka.",
    "O ya kae?",
    "Ke nako ya ho ja.",
    "Bana ba bapala kantle.",
    "Metsi a phodile."
]
english_references = [
    ["I like to read books."],
    ["Where are you going?"],
    ["It's time to eat."],
    ["Children are playing outside."],
    ["The water is cold."]
]

# Translate raw (unnormalized) South African Sesotho
print("\nTranslating raw SA Sesotho...")
raw_translations = [translator(sent)[0]['translation_text'] for sent in sa_sentences]
for sent, trans in zip(sa_sentences, raw_translations):
    print(f"  Input: {sent} -> Translation: {trans}")

# Normalize and then translate
print("\nNormalizing with SesoFix ByT5-Base...")
normalized_sa = [run_sesofix(sent) for sent in sa_sentences]
print("Translating normalized Sesotho...")
normalized_translations = [translator(sent)[0]['translation_text'] for sent in normalized_sa]
for orig, norm, trans in zip(sa_sentences, normalized_sa, normalized_translations):
    print(f"  Original SA: {orig} -> Harmonized: {norm} -> Translation: {trans}")

# Compute BLEU scores
bleu_metric = evaluate.load("bleu")
raw_bleu = bleu_metric.compute(predictions=raw_translations, references=english_references)['bleu']
norm_bleu = bleu_metric.compute(predictions=normalized_translations, references=english_references)['bleu']

print("\n--- Translation Performance Results ---")
print(f"Raw SA Translation BLEU: {raw_bleu*100:.2f}")
print(f"SesoFix Normalized BLEU: {norm_bleu*100:.2f}")
print(f"Downstream gain: {((norm_bleu - raw_bleu)*100):+.2f} BLEU")

## Section 5: Cross-Language Generalization Test

In this section, we test if SesoFix (fine-tuned on Sesotho) can generalize to neighboring Sotho-Tswana languages (such as Setswana or Sepedi/Northern Sotho) which share similar orthographic splits (like `d` vs. `l` consonant shifts). We pass Setswana samples through the model to observe morphological transfer.

In [ ]:
# Setswana sentences with regional standard orthographic variations
setswana_sentences = [
    "Ke a leboga, ke tla go bona kamoso.", 
    "Dikgomo di fula mo tshimong.",        
    "Bana ba ya sekolong gompieno."        
]

print("Running Setswana/Sepedi orthographic conversion...")
for sent in setswana_sentences:
    converted = run_sesofix(sent)
    print(f"  Input (Setswana): {sent}")
    print(f"  Output (SesoFix):  {converted}\n")